In [1]:
import os
import time
import requests
from typing import List, Dict, Any

USER_AGENT = os.getenv(
    "USER_AGENT",
    "TourGuideAI/1.0 (Learning project; contact: s0579120@htw-student.de) Mozilla/5.0 (Windows NT 10.0; Win64; x64)")

#Ort suchen mit Nominatim
def search_place(query: str) -> Dict[str, Any]:
    url = "https://nominatim.openstreetmap.org/search"
    params = {
        "q": query,
        "format": "json",
        "limit": 1,
        "addressdetails": 1,
    }
    headers = {"User-Agent": USER_AGENT}

    r = requests.get(url, params=params, headers=headers, timeout=10)
    r.raise_for_status()
    data = r.json()

    if data:
        return {
            "name": data[0].get("display_name"),
            "lat": float(data[0]["lat"]),
            "lon": float(data[0]["lon"]),
            "city": data[0]["address"].get("city") or data[0]["address"].get("town"),
            "state": data[0]["address"].get("state"),
            "country": data[0]["address"].get("country")
        }

    return None

#Museen in der Nähe suchen mit Overpass API
def get_museums_opening_hours(city_name: str):
    query = f"""
    [out:json][timeout:25];

    area["name"="Brandenburg"]["boundary"="administrative"]["admin_level"="4"]->.brandenburg;
    area(area.brandenburg)["name"="{city_name}"]["boundary"="administrative"]->.city;

    (
      node(area.city)["tourism"="museum"];
      way(area.city)["tourism"="museum"];
      relation(area.city)["tourism"="museum"];
    );
    out tags center;
    """

    r = requests.post(
        "https://overpass-api.de/api/interpreter",
        data=query,
        headers={"User-Agent": USER_AGENT},
        timeout=90
    )
    r.raise_for_status()
    data = r.json()

    museums = []
    for el in data.get("elements", []):
        tags = el.get("tags", {})
        lat = el.get("lat") or el.get("center", {}).get("lat")
        lon = el.get("lon") or el.get("center", {}).get("lon")

        museums.append({
            "name": tags.get("name"),
            "opening_hours": tags.get("opening_hours"),
            "lat": lat,
            "lon": lon,
            # "address": {
            #     "street": tags.get("addr:street"),
            #     "housenumber": tags.get("addr:housenumber"),
            #     "postcode": tags.get("addr:postcode"),
            #     "city": tags.get("addr:city")
            #}
        })

    return museums


In [2]:
# Reverse Geocoding mit Nominatim, um die Adresse zu erhalten
# def reverse_geocode(lat: float, lon: float):
#     url = "https://nominatim.openstreetmap.org/reverse"
#     params = {
#         "lat": lat,
#         "lon": lon,
#         "format": "json",
#         "addressdetails": 1
#     }
#     headers = {"User-Agent": USER_AGENT}
#
#     r = requests.get(url, params=params, headers=headers, timeout=10)
#     r.raise_for_status()
#     data = r.json()
#
#     addr = data[0].get("address", {})
#     return {
#         "street": addr.get("road"),
#         "housenumber": addr.get("house_number"),
#         "postcode": addr.get("postcode"),
#         "city": addr.get("city") or addr.get("town") or addr.get("village"),
#         "state": addr.get("state")
#     }


In [3]:
# 1) Brandenburg finden
gesuchteArea = search_place("Luebbenau/Spreewald, Brandenburg, Deutschland")


# 2) Museen in Brandenburg abrufen

if not gesuchteArea:
    print("Ort nicht gefunden.")
else:
    museums = get_museums_opening_hours('Luebbenau/Spreewald')

    for i, m in enumerate(museums[:20], start=1):
        print("Name:", m.get("name", "Unbekannt"))
        print("Öffnungszeiten:", m.get("opening_hours", "Nicht verfügbar"))
        print("-" * 40)
        # name = m.get("name", "Unbekannt")
        # opening_hours = m.get("opening_hours", "Nicht verfügbar")
        #
        # #wenn Koordinaten vorhanden:
        # if m.get('lat') is not None and m.get('lon') is not None:
        #     adresse = reverse_geocode(m['lat'], m['lon'])
        #     time.sleep(1) # Vermeidung von Rate-Limiting, sonst blockiert die API
        #
        # else:
        #     adresse = None
        #
        # print(f"--- Museum {i} ---")
        # print(f"Name: {name}")
        # print(f"Öffnungszeiten: {opening_hours}")
        #
        # if adresse:
        #     print(
        #         f"Adresse: {adresse.get('street', '')} {adresse.get('housenumber', '')}, "
        #         f"{adresse.get('postcode', '')} {adresse.get('city', '')}, "
        #         f"{adresse.get('state', '')}"
        #     )
        # else:
        #     print("Adresse: Nicht verfügbar")
        #
        #
        # print("-" * 40)



